Setting up basic cluster phrases. This allows it to be precise to what we define in the cluster, and gives more control

In [ ]:
clusters = {
    "Trauma & Mental Health": [
        "trauma-informed yoga",
        "mental health support",
        "somatic healing",
        "mindfulness for stress and anxiety",
        "yoga therapy"
    ],
    "Youth & Education": [
        "yoga in schools",
        "social emotional learning",
        "youth development programs",
        "mindfulness for students",
        "after school wellness programs"
    ],
    "Community Access & Equity": [
        "free community yoga",
        "accessible wellness programs",
        "underserved communities",
        "health equity",
        "sliding scale yoga classes"
    ],
    "assignments" : []
}


Sentiment and similarity score package

In [ ]:
!pip install sentence-transformers


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")

cluster_embeddings = {}

for cluster_name, phrases in clusters.items():
    phrase_embeddings = model.encode(phrases)
    cluster_embeddings[cluster_name] = phrase_embeddings.mean(axis=0)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
from numpy.linalg import norm

def cosine_similarity(a, b):
    return np.dot(a, b) / (norm(a) * norm(b))

def assign_cluster(grant_text):
    grant_embedding = model.encode(grant_text)

    scores = {
        name: cosine_similarity(grant_embedding, centroid)
        for name, centroid in cluster_embeddings.items()
    }

    best_cluster = max(scores, key=scores.get)
    return best_cluster, scores


Insert grant summary in grant_text and it should throw out the closest cluster it belongs to. Can be fine tuned in terms of the phrases and maybe can be connected to a vector database, but that is for later

In [ ]:
grant_text = """
We believe yoga service fosters healing, builds equity, and drives meaningful change.
Since 2007, the Give Back Yoga Foundation has supported yoga teachers and therapists bringing yoga to people with limited access to the practice—especially those facing mental, physical, or systemic hardship.

Our Yoga Service Grants help launch or expand outreach projects that make a long-term impact in underserved communities. This includes work that brings yoga into healthcare systems, schools, correctional facilities, and other institutions—advancing the recognition of yoga as a powerful tool for individual and collective well-being. We support leaders who are not only offering the practice, but transforming the systems in which it's delivered.

"""

cluster, scores = assign_cluster(grant_text)

print(cluster)
print(scores)


Community Access & Equity
{'Trauma & Mental Health': np.float32(0.5877855), 'Youth & Education': np.float32(0.5054528), 'Community Access & Equity': np.float32(0.68082476)}


In [ ]:
import json
from pathlib import Path

JSON_PATH = Path("clusters.json")

def load_data():
    if JSON_PATH.exists():
        with open(JSON_PATH, "r") as f:
            return json.load(f)
    return {"clusters": {}, "assignments": []}

def save_data(data):
    with open(JSON_PATH, "w") as f:
        json.dump(data, f, indent=2)

def log_grant(grant_text, assigned_cluster, scores):
    data = load_data()

    data["assignments"].append({
        "grant_text": grant_text.strip(),
        "assigned_cluster": assigned_cluster,
        "scores": scores
    })

    save_data(data)

def add_phrase_to_cluster(cluster_name, new_phrase):
    data = load_data()

    if cluster_name not in data["clusters"]:
        data["clusters"][cluster_name] = []

    if new_phrase not in data["clusters"][cluster_name]:
        data["clusters"][cluster_name].append(new_phrase)

    save_data(data)
